# 🚀 AstroLoc-ML on Google Colab

Open this notebook in Colab, switch to a GPU runtime, then run cells top to bottom.

**Switch to GPU:** `Runtime` → `Change runtime type` → **T4 GPU** (free tier) or **L4 / A100** (Pro).

Pipeline:
1. Clone the repo from GitHub.
2. Install Python deps (Colab already has torch + torchvision).
3. Download the HYG star catalog.
4. Smoke-test the pipeline (~30 s on GPU).
5. Fast training run (~15–25 min on T4).
6. Save the checkpoint to Google Drive so it survives the session.

## 1 · Confirm we have a GPU

In [ ]:
!nvidia-smi || echo 'No GPU. Runtime > Change runtime type > T4 GPU.'

## 2 · Clone the repo

`REPO` must be just `user/repo` — **no** `https://github.com/`, **no** `.git` suffix.
Examples: `'bachnguyennn/astro-ml'`.

In [ ]:
import pathlib
REPO = 'YOUR_USERNAME/YOUR_REPO'   # <-- change this (just user/repo)
%cd /content
!rm -rf astroloc-ml
!git clone https://github.com/{REPO}.git astroloc-ml
%cd astroloc-ml
!ls

## 3 · Install dependencies

Colab already ships with torch + torchvision + matplotlib + numpy + pandas.
We only need the smaller extras.

In [ ]:
!pip install --quiet astropy seaborn python-dotenv pyyaml piexif

## 4 · Download the HYG catalog

In [ ]:
import pathlib
pathlib.Path('data/catalogs').mkdir(parents=True, exist_ok=True)
if not pathlib.Path('data/catalogs/hygdata_v3.csv').exists():
    !curl -sL -o data/catalogs/hygdata_v3.csv \
      https://raw.githubusercontent.com/astronexus/HYG-Database/main/hyg/CURRENT/hygdata_v41.csv
!ls -lh data/catalogs/

## 5 · Smoke training (sanity check, ~30 s on T4)

In [ ]:
!python train.py --config configs/default.yaml --smoke --skip-phase3 --device cuda

## 6 · Full training run

⚠️ **Read this first.** Training is bottlenecked by **CPU**, not GPU — the synthetic
renderer generates star fields on the fly in the DataLoader, and Colab T4 instances ship
with only ~2 vCPUs. Real per-epoch time depends almost entirely on `train_samples`.

| Preset          | `train_samples` | Phase 1 | Phase 2 | Wall-clock (T4) |
| --------------- | --------------- | ------- | ------- | --------------- |
| `--smoke`       | 256             | 1 ep    | 1 ep    | ~30 s           |
| **Fast (below)**| 5,000           | 3 ep    | 6 ep    | **~15–25 min**  |
| Standard        | 20,000          | 5 ep    | 10 ep   | ~1.5–2 h        |
| Full (yaml)     | 50,000          | 5 ep    | 15 ep   | ~5–7 h          |

The **Fast preset** below gives you a real (non-random) model in 15–25 min. Switch up
only if you need the last few % of accuracy.

If you don't see the new `--train-samples` / `--epochs-phase*` / `--num-workers` flags,
run `!git pull` first — they ship in the latest commit.

In [ ]:
# Fast preset for Colab T4 — finishes in 15–25 min.
!python train.py --config configs/default.yaml --device cuda --skip-phase3 \
  --train-samples 5000 --val-samples 500 --num-workers 2 \
  --epochs-phase1 3 --epochs-phase2 6

**Want to go bigger?** Skip the cell above and run one of these instead:

```python
# Standard (~1.5–2 h) — usually beats the fast preset by a few degrees
!python train.py --config configs/default.yaml --device cuda --skip-phase3 \
  --train-samples 20000 --val-samples 1000 --num-workers 2 \
  --epochs-phase1 5 --epochs-phase2 10

# Full yaml schedule (~5–7 h)
!python train.py --config configs/default.yaml --device cuda --skip-phase3 --num-workers 2
```

## 7 · Inspect the trained checkpoint

In [ ]:
import torch, json
ckpt = torch.load('checkpoints/best.pt', map_location='cpu', weights_only=False)
print('Best val angular sep (deg):', ckpt['best_metric'])
print('History epochs:', len(ckpt.get('history', [])))
print(json.dumps(ckpt['history'][-1], indent=2, default=str))

## 8 · Regenerate README figures from the trained checkpoint

In [ ]:
!python scripts/generate_readme_figures.py

In [ ]:
from IPython.display import Image, display
for p in ['reports/figures/05_training_curves.png',
          'reports/figures/09_demo_overlay_0.png',
          'reports/figures/09_demo_overlay_1.png']:
    display(Image(p))

## 9 · Save the checkpoint to Google Drive (so it survives session restarts)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil, pathlib, time
out_dir = pathlib.Path('/content/drive/MyDrive/astroloc-ml/checkpoints')
out_dir.mkdir(parents=True, exist_ok=True)
stamp = time.strftime('%Y%m%d_%H%M%S')
shutil.copy('checkpoints/best.pt', out_dir / f'best_{stamp}.pt')
shutil.copy('reports/figures/05_training_curves.png',
            out_dir / f'training_curves_{stamp}.png')
print('Saved to', out_dir)

## 10 · Optional: download checkpoint to your laptop

In [ ]:
from google.colab import files
files.download('checkpoints/best.pt')